In [76]:
import torch

In [77]:
# Hardcoded output for the backtrace function with batch size of 2
hardcoded_paths = torch.tensor([
    [0, 1, 2, 3, 4], 
    [0, 2, 3, 4, 0]  
])

hardcoded_is_padding = torch.tensor([
    [False, False, False, False, False],  # No padding for the first batch
    [False, False, False, False, True]    # Padding at the last position for the second batch
])

# note: you can tell that last position is padding because the path goes back to 0, which
# violates the constraint of a DAG

In [78]:
# suppose 1 is the padding token
targets = torch.tensor([
    [0, 2, 2, 3, 3],  
    [3, 2, 1, 3, 1]   
])

In [79]:
torch.manual_seed(0)

In [80]:
batch_size = 2
num_tokens = 5
vocab_size = 4
upsample_factor = 2

In [81]:
simulated_logits = torch.randn(batch_size, num_tokens * upsample_factor, vocab_size)

In [82]:
simulated_logits

tensor([[[-1.1258, -1.1524, -0.2506, -0.4339],
         [ 0.8487,  0.6920, -0.3160, -2.1152],
         [ 0.3223, -1.2633,  0.3500,  0.3081],
         [ 0.1198,  1.2377,  1.1168, -0.2473],
         [-1.3527, -1.6959,  0.5667,  0.7935],
         [ 0.5988, -1.5551, -0.3414,  1.8530],
         [ 0.7502, -0.5855, -0.1734,  0.1835],
         [ 1.3894,  1.5863,  0.9463, -0.8437],
         [-0.6136,  0.0316, -0.4927,  0.2484],
         [ 0.4397,  0.1124,  0.6408,  0.4412]],

        [[-0.1023,  0.7924, -0.2897,  0.0525],
         [ 0.5229,  2.3022, -1.4689, -1.5867],
         [-0.6731,  0.8728,  1.0554,  0.1778],
         [-0.2303, -0.3918,  0.5433, -0.3952],
         [-0.4462,  0.7440,  1.5210,  3.4105],
         [-1.5312, -1.2341,  1.8197, -0.5515],
         [-0.5692,  0.9200,  1.1108,  1.2899],
         [-1.4782,  2.5672, -0.4731,  0.3356],
         [-1.6293, -0.5497, -0.4798, -0.4997],
         [-1.0670,  1.1149, -0.1407,  0.8058]]])

In [83]:
# for the simulated logits, we only care about the positions specified by the paths
simulated_paths = torch.gather(
    simulated_logits, 1, hardcoded_paths.unsqueeze(-1).expand(-1, -1, vocab_size)
)

In [84]:
simulated_paths, hardcoded_paths

(tensor([[[-1.1258, -1.1524, -0.2506, -0.4339],
          [ 0.8487,  0.6920, -0.3160, -2.1152],
          [ 0.3223, -1.2633,  0.3500,  0.3081],
          [ 0.1198,  1.2377,  1.1168, -0.2473],
          [-1.3527, -1.6959,  0.5667,  0.7935]],
 
         [[-0.1023,  0.7924, -0.2897,  0.0525],
          [-0.6731,  0.8728,  1.0554,  0.1778],
          [-0.2303, -0.3918,  0.5433, -0.3952],
          [-0.4462,  0.7440,  1.5210,  3.4105],
          [-0.1023,  0.7924, -0.2897,  0.0525]]]),
 tensor([[0, 1, 2, 3, 4],
         [0, 2, 3, 4, 0]]))

In [85]:
chosen_tokens = torch.argmax(simulated_paths, dim=-1)  # Simulate the argmax operation

In [86]:
chosen_tokens

tensor([[2, 0, 2, 1, 3],
        [1, 2, 2, 3, 1]])

In [87]:
matches = chosen_tokens == targets

In [95]:
matches

tensor([[False, False,  True, False,  True],
        [False,  True, False,  True,  True]])

In [96]:
# automatically label any position that is a padding token as not a match
matches = matches & ~hardcoded_is_padding

In [97]:
matches

tensor([[False, False,  True, False,  True],
        [False,  True, False,  True, False]])

In [ ]:
# get number of matches
num_matches = matches.sum()
# get number of padding tokens
num_padding = hardcoded_is_padding.sum()

In [89]:
num_matches, num_padding

(tensor(5), tensor(1))

In [90]:
batch_size, seq_len = targets.shape

In [91]:
total = batch_size * seq_len

In [92]:
total

10

In [93]:
num_matches / (total - num_padding)

tensor(0.5556)

In [101]:
probs = torch.tensor([[0.1, 0.7, 0.2], [0.3, 0.4, 0.3]])

# Log probabilities
log_probs = torch.log(probs)

# Argmax in probability space
most_likely_probs = torch.argmax(probs, dim=-1)

# Argmax in log space
most_likely_log_probs = torch.argmax(log_probs, dim=-1)

print(most_likely_probs)       # tensor([1, 1])
print(most_likely_log_probs)   # tensor([1, 1]) (same result)

tensor([1, 1])
tensor([1, 1])


In [102]:
log_probs

tensor([[-2.3026, -0.3567, -1.6094],
        [-1.2040, -0.9163, -1.2040]])